# Fetching Census Survey Data

morpc-census connects to the US Census Bureau API and returns survey data as a long-format DataFrame ready for analysis. Every query is defined by four required parameters and two optional ones:

**Required**
- `survey_table` — the dataset endpoint, e.g. `'acs/acs5'` or `'dec/pl'`
- `year` — the vintage year, e.g. `2023`
- `group` — the variable group code, e.g. `'B01001'` (sex by age)
- `scope` — the geographic extent, e.g. `'region15'` or `'franklin'`

**Optional**
- `sumlevel` — resolution within the scope, e.g. `'tract'` or `'county'`
- `variables` — a subset of variables from the group

> **Network required** — all cells below make live calls to the Census API.

In [39]:
from morpc_census import (
    CensusAPI, DimensionTable,
    IMPLEMENTED_ENDPOINTS, get_all_avail_endpoints,
    get_table_groups, get_group_variables,
    SCOPES, PSEUDOS,
)
import pandas as pd

## 1. What surveys are available?

`IMPLEMENTED_ENDPOINTS` lists every survey/table the package supports. For any endpoint, `get_all_avail_endpoints()` returns all vintages the Census API exposes (cached after the first call).

In [40]:
# Surveys the package supports
IMPLEMENTED_ENDPOINTS

['acs/acs1',
 'acs/acs1/profile',
 'acs/acs1/subject',
 'acs/acs5',
 'acs/acs5/profile',
 'acs/acs5/subject',
 'dec/pl',
 'dec/dhc',
 'dec/ddhca',
 'dec/ddhcb',
 'dec/sf1',
 'dec/sf2',
 'dec/sf3',
 'geoinfo']

In [41]:
# Available vintage years for ACS 5-year
get_all_avail_endpoints()['acs/acs5']

[2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024]

## 2. What variables are in a group?

`get_table_groups()` returns every group in a survey with its description. `get_group_variables()` drills into a specific group to show individual variable codes and labels.

In [42]:
# Browse available groups — show one example description
groups = get_table_groups('acs/acs5', 2023)
{k: v['description'] for k, v in list(groups.items())[:5]}

{'B01001': 'Sex by Age',
 'B01001A': 'Sex by Age (White Alone)',
 'B01001B': 'Sex by Age (Black or African American Alone)',
 'B01001C': 'Sex by Age (American Indian and Alaska Native Alone)',
 'B01001D': 'Sex by Age (Asian Alone)'}

In [43]:
# Inspect variables in group B01001 (sex by age)
vars_b01001 = get_group_variables('acs/acs5', 2023, 'B01001')
{k: v['label'] for k, v in list(vars_b01001.items())[:8]}

{'B01001_001E': 'Estimate!!Total:',
 'B01001_001EA': 'Annotation of Estimate!!Total:',
 'B01001_001M': 'Margin of Error!!Total:',
 'B01001_001MA': 'Annotation of Margin of Error!!Total:',
 'B01001_002E': 'Estimate!!Total:!!Male:',
 'B01001_002EA': 'Annotation of Estimate!!Total:!!Male:',
 'B01001_002M': 'Margin of Error!!Total:!!Male:',
 'B01001_002MA': 'Annotation of Margin of Error!!Total:!!Male:'}

## 3. Choosing a scope and sumlevel

`SCOPES` names every built-in geographic extent. `PSEUDOS` shows which child sumlevels can be fetched within each parent sumlevel — the key is the parent's summary level code and the values are the allowed child codes.

In [44]:
# Available scope names
list(SCOPES.keys())

['us',
 'columbuscbsa',
 'alabama',
 'alaska',
 'arizona',
 'arkansas',
 'california',
 'colorado',
 'connecticut',
 'delaware',
 'district of columbia',
 'florida',
 'georgia',
 'hawaii',
 'idaho',
 'illinois',
 'indiana',
 'iowa',
 'kansas',
 'kentucky',
 'louisiana',
 'maine',
 'maryland',
 'massachusetts',
 'michigan',
 'minnesota',
 'mississippi',
 'missouri',
 'montana',
 'nebraska',
 'nevada',
 'new hampshire',
 'new jersey',
 'new mexico',
 'new york',
 'north carolina',
 'north dakota',
 'ohio',
 'oklahoma',
 'oregon',
 'pennsylvania',
 'rhode island',
 'south carolina',
 'south dakota',
 'tennessee',
 'texas',
 'utah',
 'vermont',
 'virginia',
 'washington',
 'west virginia',
 'wisconsin',
 'wyoming',
 'puerto rico',
 'lake',
 'hancock',
 'allen',
 'morgan',
 'portage',
 'butler',
 'fayette',
 'vinton',
 'paulding',
 'scioto',
 'fulton',
 'henry',
 'logan',
 'brown',
 'monroe',
 'trumbull',
 'pike',
 'pickaway',
 'muskingum',
 'lawrence',
 'adams',
 'crawford',
 'guernsey',
 

In [45]:
# Child sumlevels available when the scope is county-level (050)
PSEUDOS['050']

['0600000',
 '06V0000',
 '1000000',
 '1400000',
 '1500000',
 '1600000',
 '7000000',
 '8600000',
 '8710000']

## 4. Fetching data

`CensusAPI` validates parameters, builds the request, fetches the data, and immediately transforms it into a long-format `LONG` table. The raw wide response is also stored as `DATA`.

In [46]:
# Sex by age for all counties in the 15-county MORPC region
b01001 = CensusAPI('acs/acs5', 2023, 'B01001', scope='region15')
b01001.DATA

,index,B01001_001E,B01001_001EA,B01001_001M,B01001_001MA,B01001_002E,B01001_002EA,B01001_002M,B01001_002MA,B01001_003E,...,B01001_048M,B01001_048MA,B01001_049E,B01001_049EA,B01001_049M,B01001_049MA,GEO_ID,NAME,state,county
0,0,221160,NaN,-555555555,*****,110668,NaN,33,NaN,6456,...,290,NaN,2009,NaN,352,NaN,0500000US39041,"Delaware County, Ohio",39,41
1,1,161289,NaN,-555555555,*****,80584,NaN,128,NaN,4690,...,241,NaN,1892,NaN,225,NaN,0500000US39045,"Fairfield County, Ohio",39,45
2,2,28880,NaN,-555555555,*****,14271,NaN,163,NaN,778,...,106,NaN,352,NaN,91,NaN,0500000US39047,"Fayette County, Ohio",39,47
3,3,1321635,NaN,-555555555,*****,649003,NaN,61,NaN,44950,...,692,NaN,11174,NaN,747,NaN,0500000US39049,"Franklin County, Ohio",39,49
4,4,27938,NaN,-555555555,*****,14022,NaN,110,NaN,808,...,102,NaN,248,NaN,87,NaN,0500000US39073,"Hocking County, Ohio",39,73
5,5,62888,NaN,-555555555,*****,31021,NaN,150,NaN,1943,...,175,NaN,895,NaN,160,NaN,0500000US39083,"Knox County, Ohio",39,83
6,6,180311,NaN,-555555555,*****,89196,NaN,116,NaN,5355,...,266,NaN,2124,NaN,256,NaN,0500000US39089,"Licking County, Ohio",39,89
7,7,46140,NaN,-555555555,*****,23206,NaN,100,NaN,1435,...,158,NaN,464,NaN,107,NaN,0500000US39091,"Logan County, Ohio",39,91
8,8,44126,NaN,-555555555,*****,24262,NaN,160,NaN,1181,...,126,NaN,525,NaN,130,NaN,0500000US39097,"Madison County, Ohio",39,97
9,9,65145,NaN,-555555555,*****,35012,NaN,168,NaN,1943,...,185,NaN,772,NaN,158,NaN,0500000US39101,"Marion County, Ohio",39,101


In [47]:
# The long-format result — one row per geography × variable × value type
b01001.LONG

,GEO_ID,NAME,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:,B01001_001,221160,-555555555
25,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Male:,B01001_002,110668,33
48,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!Under 5 years,B01001_003,6456,6
37,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!5 to 9 years,B01001_004,7560,466
26,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!10 to 14 years,B01001_005,9200,465
...,...,...,...,...,...,...,...,...,...,...
705,0500000US39159,"Union County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!67 to 69 years,B01001_045,914,183
706,0500000US39159,"Union County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!70 to 74 years,B01001_046,1201,211
707,0500000US39159,"Union County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!75 to 79 years,B01001_047,738,161
708,0500000US39159,"Union County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!80 to 84 years,B01001_048,571,129


In [48]:
# Adding sumlevel='tract' fetches all census tracts within the region's counties
b05006 = CensusAPI('acs/acs5', 2023, 'B05006', scope='region15', sumlevel='tract')
b05006.LONG

,GEO_ID,NAME,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,1400000US39041010100,Census Tract 101; Delaware County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:,B05006_001,214,98
125,1400000US39041010100,Census Tract 101; Delaware County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:,B05006_002,23,38
145,1400000US39041010100,Census Tract 101; Delaware County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:,B05006_003,0,18
146,1400000US39041010100,Census Tract 101; Delaware County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:!!Denmark,B05006_004,0,18
147,1400000US39041010100,Census Tract 101; Delaware County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:!!Ireland,B05006_005,0,18
...,...,...,...,...,...,...,...,...,...,...
103669,1400000US39159050702,Census Tract 507.02; Union County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Latin America:!!South Ameri...,B05006_174,0,13
103666,1400000US39159050702,Census Tract 507.02; Union County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Latin America:!!South Ameri...,B05006_175,0,13
103670,1400000US39159050702,Census Tract 507.02; Union County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Northern America:,B05006_176,0,13
103671,1400000US39159050702,Census Tract 507.02; Union County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Northern America:!!Canada,B05006_177,0,13


In [49]:
# The scope and sumlevel are stored on the object for inspection
print(b05006.SCOPE, b05006.SUMLEVEL)
b05006.scope_obj    # the Scope object for this dataset's geographic extent

region15 tract


Scope(name='region15', for_param='county:041,045,049,089,097,129,159,083,101,117,047,073,091,127,141', in_param='state:39')

In [50]:
# GEO_ID values parsed as GeoIDFQ objects — sumlevel, variant, geocomp, parts
b05006.geoidfqs[:3]

[GeoIDFQ(sumlevel='140', variant='00', geocomp='00', state='39', county='041', tract='010100'),
 GeoIDFQ(sumlevel='140', variant='00', geocomp='00', state='39', county='041', tract='010200'),
 GeoIDFQ(sumlevel='140', variant='00', geocomp='00', state='39', county='041', tract='010420')]

## 5. Analyzing with DimensionTable

`DimensionTable` pivots long-format data into wide and percentage tables — the same layout the Census Bureau uses in its published tables, with variable labels as a hierarchical index.

In [51]:
# Wide format: variable labels as rows, geographies as columns
dim = DimensionTable(b01001.LONG)
dim.wide()

estimate                   moe  \
GEO_ID                                  0500000US39041        0500000US39041   
NAME                             Delaware County, Ohio Delaware County, Ohio   
reference_period                                  2023                  2023   
survey                                        acs/acs5              acs/acs5   
concept                                     Sex by age            Sex by age   
universe                              Total population      Total population   
0      1       2                                                               
Total:                                          221160            -555555555   
       Male:                                    110668                    33   
               Under 5 years                      6456                     6   
               5 to 9 years                       7560                   466   
               10 to 14 years                     9200                   465   
               15 to 17 years                     5478                    22   
               18 and 19 years                    3068                   176   
               20 years                           1337                   298   
               21 years                           1490                   305   
               22 to 24 years                     3434                   362   
               25 to 29 years                     4556                   140   
               30 to 34 years                     5846                    20   
               35 to 39 years                     7477                   606   
               40 to 44 years                     9403                   609   
               45 to 49 years                     8681                    19   
               50 to 54 years                     8309                    26   
               55 to 59 years                     7351                   547   
               60 and 61 years                    2678                   457   
               62 to 64 years                     3255                   382   
               65 and 66 years                    2304                   346   
               67 to 69 years                     2872                   310   
               70 to 74 years                     4734                   364   
               75 to 79 years                     2642                   284   
               80 to 84 years                     1212                   179   
               85 years and over                  1325                   273   
       Female:                                  110492                    33   
               Under 5 years                      6078                    23   
               5 to 9 years                       7775                   413   
               10 to 14 years                     8150                   411   
               15 to 17 years                     5177                    19   
               18 and 19 years                    2782                    97   
               20 years                           1353                   306   
               21 years                           1370                   325   
               22 to 24 years                     3133                   410   
               25 to 29 years                     4701                   136   
               30 to 34 years                     6186                    85   
               35 to 39 years                     8321                   550   
               40 to 44 years                     8575                   547   
               45 to 49 years                     8251                    15   
               50 to 54 years                     7863                    15   
               55 to 59 years                     7343                   397   
               60 and 61 years                    2459                   309   
               62 to 64 years                 

## 6. Time series

`DimensionTable` accepts any long-format DataFrame with the same schema. Concatenating `LONG` from multiple years produces a time-series table where `reference_period` becomes a column axis.

In [53]:
# Fetch the same group for an earlier vintage
b01001_2018 = CensusAPI('acs/acs5', 2018, 'B01001', scope='region15')

# Stack the two years and build a dimension table
long_ts = pd.concat([b01001_2018.LONG, b01001.LONG])
DimensionTable(long_ts).wide()

estimate                   \
GEO_ID                                  0500000US39041                    
NAME                             Delaware County, Ohio                    
reference_period                                  2018             2023   
survey                                        acs/acs5         acs/acs5   
concept                                     Sex by age       Sex by age   
universe                              Total population Total population   
0      1       2                                                          
Total:                                          197008           221160   
       Male:                                     97504           110668   
               Under 5 years                      6273             6456   
               5 to 9 years                       7752             7560   
               10 to 14 years                     8435             9200   
               15 to 17 years                     4731             5478   
               18 and 19 years                    2531             3068   
               20 years                           1515             1337   
               21 years                           1000             1490   
               22 to 24 years                     2824             3434   
               25 to 29 years                     3911             4556   
               30 to 34 years                     5349             5846   
               35 to 39 years                     7256             7477   
               40 to 44 years                     7690             9403   
               45 to 49 years                     8006             8681   
               50 to 54 years                     7175             8309   
               55 to 59 years                     6484             7351   
               60 and 61 years                    2254             2678   
               62 to 64 years                     2915             3255   
               65 and 66 years                    1892             2304   
               67 to 69 years                     2676             2872   
               70 to 74 years                     2946             4734   
               75 to 79 years                     1655             2642   
               80 to 84 years                     1296             1212   
               85 years and over                   938             1325   
       Female:                                   99504           110492   
               Under 5 years                      6112             6078   
               5 to 9 years                       7693             7775   
               10 to 14 years                     7624             8150   
               15 to 17 years                     4612             5177   
               18 and 19 years                    2375             2782   
               20 years                           1086             1353   
               21 years                            973             1370   
               22 to 24 years                     2928             3133   
               25 to 29 years                     4177             4701   
               30 to 34 years                     6046             6186   
               35 to 39 years                     7589             8321   
               40 to 44 years                     7861             8575   
               45 to 49 years                     7741             8251   
               50 to 54 years                     7104             7863   
               55 to 59 years                     6412             7343   
               60 and 61 years                    2287             2459   
               62 to 64 years                     3262             3557   
               65 and 66 years                    1653             2722   
               67 to 69 years                     2828             2937   
               70 to 74 years                     3630             4738

## 7. Saving data

`CensusAPI.save()` writes three files to the output directory:
- `{name}.long.csv` — the long-format data
- `{name}.schema.yaml` — a frictionless Schema
- `{name}.resource.yaml` — a frictionless Resource descriptor pointing at the above two

In [54]:
import os

b01001.save('./temp_data')

print(b01001.FILENAME)
os.path.exists(f'./temp_data/{b01001.FILENAME}')

census-acs-acs5-2023-region15-b01001.long.csv


True

In [56]:
from frictionless import Resource, Schema

resource = Resource('temp_data/census-acs-acs5-2023-region15-b01001.resource.yaml')
resource

{'name': 'census-acs-acs5-2023-region15-b01001',
 'type': 'table',
 'title': '2023 Sex by Age for region15',
 'description': 'Census API data for B01001: Sex by Age from acs/acs5 in 2023 '
                'for region15.',
 'sources': [{'title': 'US Census Bureau API',
              'path': 'https://api.census.gov/data/2023/acs/acs5?',
              '_params': {'get': 'group(B01001)',
                          'for': 'county:041,045,049,089,097,129,159,083,101,117,047,073,091,127,141',
                          'in': 'state:39'}}],
 'path': 'census-acs-acs5-2023-region15-b01001.long.csv',
 'scheme': 'file',
 'format': 'csv',
 'mediatype': 'text/csv',
 'schema': 'census-acs-acs5-2023-region15-b01001.schema.yaml'}

In [57]:
schema = Schema('temp_data/census-acs-acs5-2023-region15-b01001.schema.yaml')
schema

{'fields': [{'name': 'GEO_ID',
             'type': 'string',
             'description': 'Census geography identifier'},
            {'name': 'NAME', 'type': 'string', 'description': 'Geography name'},
            {'name': 'reference_period',
             'type': 'integer',
             'description': 'Reference year'},
            {'name': 'survey',
             'type': 'string',
             'description': 'Census survey endpoint'},
            {'name': 'concept',
             'type': 'string',
             'description': 'Table concept description'},
            {'name': 'universe',
             'type': 'string',
             'description': 'Universe for the table'},
            {'name': 'variable_label',
             'type': 'string',
             'description': 'Human-readable variable label'},
            {'name': 'variable',
             'type': 'string',
             'description': 'Base variable code'},
            {'name': 'estimate',
             'type': 'number',
         